# Future Work 2 — Improved Black-box Defense: IDS-Anta++
Based on: IDS-Anta (Barik & Misra, 2024)

In [ ]:
!pip install adversarial-robustness-toolbox --quiet

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, precision_recall_fscore_support

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical

# NOTE: Do NOT call disable_eager_execution
print('TF version:', tf.__version__)
print('All imports successful.')

In [ ]:
# AUTO DETECT FILE PATHS
all_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        all_files.append(full)
        print(full)

PATH_2017 = [f for f in all_files if '2017' in f][0]
PATH_2018 = [f for f in all_files if '2018' in f][0]
PATH_2019 = [f for f in all_files if '2019' in f][0]

print('\n2017:', PATH_2017)
print('2018:', PATH_2018)
print('2019:', PATH_2019)

In [ ]:
# Preprocessing
def preprocess(path, label_col, benign_value, drop_cols=[]):
    df = pd.read_csv(path)
    print(f'Loaded: {df.shape}')
    raw = df[label_col].astype(str).str.strip()
    y   = np.where(raw == benign_value.strip(), 0, 1)
    drop = [label_col] + drop_cols
    df   = df.drop(columns=[c for c in drop if c in df.columns])
    df   = df.select_dtypes(include=[np.float64, np.int64])
    df   = df.fillna(df.mean())
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    y    = y[:len(df)]
    X    = StandardScaler().fit_transform(df)
    X    = TruncatedSVD(n_components=min(10, X.shape[1]-1)).fit_transform(X)
    print(f'Features: {X.shape} | Benign: {(y==0).sum()} | Attack: {(y==1).sum()}')
    return X.astype(np.float32), y.astype(np.int32)

print('Preprocessing ready.')

In [ ]:
# DNN builder
def build_dnn(input_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(64,  activation='relu'),
        keras.layers.Dense(2,   activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Evaluation
def evaluate_model(y_true, y_pred, y_proba, model_name, attack_name='ZOO'):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0)
    auc = roc_auc_score(y_true, y_proba) if y_proba is not None else 0.0
    print(f'  Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {p:.4f} | Recall: {r:.4f} | AUC: {auc:.4f}')
    return {'Model': model_name, 'Attack': attack_name,
            'Accuracy': round(acc,4), 'F1': round(f1,4),
            'Precision': round(p,4), 'Recall': round(r,4),
            'Detection Rate': round(r,4), 'AUC': round(auc,4)}

print('Helpers ready.')

In [ ]:
# IDS-Anta++ — Proposed improved defense
class IDSAntaPlus:
    def __init__(self, rf, svm, lr, dnn):
        self.rf       = rf
        self.svm      = svm
        self.lr       = lr
        self.dnn      = dnn
        self.detector = IsolationForest(
            contamination=0.05, n_estimators=100, random_state=42)

    def fit(self, X_clean):
        self.detector.fit(X_clean)
        print('IDS-Anta++ detection layer trained.')

    def _smooth(self, X, window=3):
        Xs = np.copy(X)
        for i in range(X.shape[1]):
            Xs[:, i] = pd.Series(X[:, i]).rolling(
                window=window, min_periods=1, center=True).median().values
        return Xs.astype(np.float32)

    def predict(self, X):
        is_adv = (self.detector.predict(X) == -1)
        X_s    = self._smooth(X)
        v_rf   = self.rf.predict(X_s)
        v_svm  = self.svm.predict(X_s)
        v_lr   = self.lr.predict(X_s)
        v_dnn  = np.argmax(self.dnn.predict(X_s, verbose=0), axis=1)
        votes  = np.stack([v_rf, v_svm, v_lr, v_dnn], axis=1)
        voted  = np.apply_along_axis(
            lambda row: np.bincount(row, minlength=2).argmax(), 1, votes)
        return np.where(is_adv, 1, voted)

print('IDS-Anta++ defined.')

In [ ]:
# Experiment function
def run_experiment(X, y, dataset_name):
    print(f'\n{"="*60}\nDATASET: {dataset_name}\n{"="*60}')

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    results = []

    print('Training RF, SVM, LR...')
    rf  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    svm = SVC(kernel='rbf', probability=True, random_state=42)
    lr  = LogisticRegression(max_iter=1000, random_state=42)
    rf.fit(X_train, y_train)
    svm.fit(X_train, y_train)
    lr.fit(X_train, y_train)

    print('Training DNN...')
    dnn = build_dnn(X_train.shape[1])
    dnn.fit(X_train, to_categorical(y_train, 2),
            epochs=20, batch_size=64, validation_split=0.1, verbose=1)
    print('All classifiers trained.')

    print('Training IDS-Anta++ defense...')
    ids_plus = IDSAntaPlus(rf, svm, lr, dnn)
    ids_plus.fit(X_train)

    # ART wrapper using TensorFlowV2Classifier
    from art.estimators.classification import TensorFlowV2Classifier
    loss_fn = tf.keras.losses.CategoricalCrossentropy()
    art_dnn = TensorFlowV2Classifier(
        model=dnn, nb_classes=2,
        input_shape=(X_train.shape[1],),
        loss_object=loss_fn,
        clip_values=(float(X_train.min()), float(X_train.max()))
    )

    print('\nGenerating ZOO adversarial examples (takes ~10 mins)...')
    from art.attacks.evasion import ZooAttack
    n_zoo = min(200, len(X_test))
    X_zoo = ZooAttack(
        classifier=art_dnn, confidence=0.0, targeted=False,
        learning_rate=1e-1, max_iter=100, binary_search_steps=10,
        abort_early=True, verbose=False
    ).generate(X_test[:n_zoo])
    y_zoo = y_test[:n_zoo]
    print(f'ZOO done: {X_zoo.shape}')

    print('RF (no defense):')
    results.append(evaluate_model(y_zoo, rf.predict(X_zoo), rf.predict_proba(X_zoo)[:,1], 'RF (no defense)'))
    print('SVM (no defense):')
    results.append(evaluate_model(y_zoo, svm.predict(X_zoo), svm.predict_proba(X_zoo)[:,1], 'SVM (no defense)'))
    print('LR (no defense):')
    results.append(evaluate_model(y_zoo, lr.predict(X_zoo), lr.predict_proba(X_zoo)[:,1], 'LR (no defense)'))
    print('DNN (no defense):')
    p = dnn.predict(X_zoo, verbose=0)
    results.append(evaluate_model(y_zoo, np.argmax(p,1), p[:,1], 'DNN (no defense)'))

    print('RF + Input Smoothing:')
    X_smooth = ids_plus._smooth(X_zoo)
    results.append(evaluate_model(y_zoo, rf.predict(X_smooth), rf.predict_proba(X_smooth)[:,1], 'RF + Smoothing'))

    print('Ensemble Majority Voting:')
    votes = np.stack([
        rf.predict(X_smooth), svm.predict(X_smooth),
        lr.predict(X_smooth),
        np.argmax(dnn.predict(X_smooth, verbose=0), axis=1)
    ], axis=1)
    voted = np.apply_along_axis(lambda r: np.bincount(r, minlength=2).argmax(), 1, votes)
    results.append(evaluate_model(y_zoo, voted, None, 'Ensemble Voting'))

    print('IDS-Anta++ (Proposed):')
    results.append(evaluate_model(y_zoo, ids_plus.predict(X_zoo), None, 'IDS-Anta++ (Proposed)'))

    return pd.DataFrame(results)

print('Experiment function ready.')

In [ ]:
X_2017, y_2017 = preprocess(PATH_2017, label_col='Label', benign_value='BENIGN')
results_2017 = run_experiment(X_2017, y_2017, 'CIC-IDS-2017')
results_2017.to_csv('FW2_results_2017.csv', index=False)
display(results_2017)

In [ ]:
X_2018, y_2018 = preprocess(PATH_2018, label_col='Label', benign_value='Benign',
                              drop_cols=['Dst Port','Protocol','Timestamp'])
results_2018 = run_experiment(X_2018, y_2018, 'CEC-CIC-IDS-2018')
results_2018.to_csv('FW2_results_2018.csv', index=False)
display(results_2018)

In [ ]:
X_2019, y_2019 = preprocess(PATH_2019, label_col=' Label', benign_value='BENIGN',
                              drop_cols=['Unnamed: 0'])
results_2019 = run_experiment(X_2019, y_2019, 'CIC-DDoS-2019')
results_2019.to_csv('FW2_results_2019.csv', index=False)
display(results_2019)

In [ ]:
for name, df_r in [('2017', results_2017), ('2018', results_2018), ('2019', results_2019)]:
    print(f'\n--- CIC-IDS-{name} ---')
    print(df_r.to_string(index=False))

In [ ]:
for year, df_r in [('2017', results_2017), ('2018', results_2018), ('2019', results_2019)]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metric in zip(axes, ['Accuracy', 'F1']):
        models = df_r['Model'].tolist()
        values = df_r[metric].tolist()
        colors = ['#d9534f' if 'no defense' in m
                  else '#f0ad4e' if 'Smooth' in m or 'Voting' in m
                  else '#5cb85c' for m in models]
        bars = ax.barh(models, values, color=colors)
        ax.set_xlim(0, 1.1)
        ax.set_title(f'{metric} — CIC-IDS-{year}', fontsize=11)
        for bar, val in zip(bars, values):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=9)
    plt.suptitle(f'IDS-Anta++ vs Baselines (ZOO Attack) — CIC-IDS-{year}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'FW2_{year}_defense.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: FW2_{year}_defense.png')

print('All done! Download CSV and PNG from the Output tab on the right.')